In [0]:
# All package installs in one place — numpy<2 must come last to downgrade
%pip install ecmwf-api-client cdsapi cftime bottleneck microsoft-aurora
%pip install "numpy<2" "xarray<2024.7" netcdf4 --force-reinstall
dbutils.library.restartPython()

In [0]:
import sys
print(sys.version)

import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)

In [0]:
import os

checkpoint_path = "/dbfs/models/aurora-0.1-finetuned.ckpt"

if os.path.exists(checkpoint_path):
    print(f"Checkpoint file already exists at {checkpoint_path}, skipping download.")
else:
    print("Downloading checkpoint file...")
    # Create directory if it doesn't exist
    os.makedirs("/dbfs/models", exist_ok=True)
    !wget https://huggingface.co/microsoft/aurora/resolve/main/aurora-0.1-finetuned.ckpt -P /dbfs/models/
    print("Download complete.")

In [0]:
import os
import psutil

# Check available memory before loading
memory_info = psutil.virtual_memory()
available_gb = memory_info.available / (1024**3)
print(f"Available memory: {available_gb:.2f} GB")

if available_gb < 10:
    print("⚠️ Warning: Less than 10GB of available memory. Model loading may fail or be very slow.")
    print("Consider using a larger cluster or GPU-enabled compute.")

checkpoint_path = "/dbfs/models/aurora-0.1-finetuned.ckpt"

if not os.path.exists(checkpoint_path):
    print(f"Error: Checkpoint file not found at {checkpoint_path}")
    print("Please run the previous cell to download the checkpoint first.")
else:
    try:
        from aurora import AuroraHighRes
        print("Aurora HighRes model")
        model = AuroraHighRes()
        print("Loading model checkpoint...")
        model.load_checkpoint_local(checkpoint_path)
        print("Checkpoint loaded")
        model.eval() # Disable training specific behaviour like dropout
        print("✓ Model loaded successfully")
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        print("\nTroubleshooting tips:")
        print("1. Ensure you have sufficient memory (16GB+ recommended)")
        print("2. Consider using a GPU-enabled cluster for better performance")
        print("3. Close other notebooks or restart the cluster if memory is full")

In [0]:
# ------------------------------
# Imports
# ------------------------------
import os
import xarray as xr
import numpy as np
import torch
import pickle
from aurora import AuroraHighRes, Batch, Metadata, rollout
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------
# Configuration (SINGLE TIMESTEP)
# ------------------------------
sfc_file = "/dbfs/tmp/ec_data_sample_sfc.grib"
pl_file  = "/dbfs/tmp/ec_data_sample_pl.grib"
output_dir = "/dbfs/tmp/aurora_forecasts"

os.makedirs(output_dir, exist_ok=True)

In [0]:
# ------------------------------
# Helper functions
# ------------------------------
def fix_coordinates(ds):
    lon_values = ds.longitude.values.astype(np.float32)
    new_lons = np.mod(lon_values, 360)
    new_lons[new_lons == 360] = 0.0
    ds = ds.assign_coords(longitude=new_lons)
    ds = ds.sortby("longitude")
    return ds

def lon_360_to_180(lon):
    lon = lon.copy()
    lon[lon > 180] -= 360
    return lon

In [0]:
# ------------------------------
# Load static variables
# ------------------------------
print("Downloading static variables...")
static_path = hf_hub_download(repo_id="microsoft/aurora", filename="aurora-0.1-static.pickle")
with open(static_path, "rb") as f:
    static_vars = pickle.load(f)
print("Static variables loaded.")


In [0]:
# ------------------------------
# Load GRIB files (via CDO → NetCDF, read with netCDF4 to bypass eccodes crash)
# ------------------------------
import subprocess
import netCDF4

# Write NetCDF to local disk (/dbfs FUSE doesn't support CDO's file creation)
sfc_nc = "/tmp/ec_data_sample_sfc.nc"

# Convert surface GRIB to NetCDF
if not os.path.exists(sfc_nc):
    print("Converting surface GRIB → NetCDF...")
    subprocess.run(["cdo", "-f", "nc", "copy", sfc_file, sfc_nc], check=True)
    print("✓ Surface conversion complete")
else:
    print(f"[SKIP] Surface NetCDF already exists: {sfc_nc}")

# Convert pressure-level GRIB per variable (all-at-once crashes CDO)
pl_vars_list = ["t", "q", "z", "u", "v"]
for var in pl_vars_list:
    out = f"/tmp/pl_{var}.nc"
    if not os.path.exists(out):
        print(f"Converting PL var {var} → NetCDF...")
        subprocess.run(["cdo", "-f", "nc", f"selname,{var}", pl_file, out], check=True)

# --- Read surface NetCDF with netCDF4 (avoids xarray backend discovery) ---
print("Loading surface data...")
nc = netCDF4.Dataset(sfc_nc, "r")
sfc_coords = {
    "longitude": nc.variables["lon"][:],
    "latitude": nc.variables["lat"][:],
}

# Auto-extract timestamp from the data
time_var = nc.variables["time"]
time_decoded = netCDF4.num2date(time_var[:], units=time_var.units,
                                calendar=getattr(time_var, "calendar", "standard"))
data_time = np.datetime64(str(time_decoded[0]))
print(f"  Data timestamp: {data_time}")

sfc_data = {}
# CDO names → cfgrib names (so Cell 12's rename from t2m→2t works)
cdo_to_cfgrib = {"2t": "t2m", "10u": "u10", "10v": "v10", "msl": "msl"}
for cdo_name, cfgrib_name in cdo_to_cfgrib.items():
    sfc_data[cfgrib_name] = (["latitude", "longitude"], nc.variables[cdo_name][0, :, :])
nc.close()
sfc_ds = xr.Dataset(data_vars=sfc_data, coords=sfc_coords)
print(f"  Variables: {list(sfc_ds.data_vars)}, Dims: {dict(sfc_ds.dims)}")

# --- Read pressure-level NetCDF files with netCDF4 ---
print("Loading pressure-level data...")
pl_data = {}
for var in pl_vars_list:
    nc = netCDF4.Dataset(f"/tmp/pl_{var}.nc", "r")
    if "isobaricInhPa" not in pl_data:
        # First file: extract shared coordinates, convert plev Pa → hPa
        pl_coords = {
            "longitude": nc.variables["lon"][:],
            "latitude": nc.variables["lat"][:],
            "isobaricInhPa": nc.variables["plev"][:] / 100.0,
        }
    pl_data[var] = (["isobaricInhPa", "latitude", "longitude"], nc.variables[var][0, :, :, :])
    nc.close()
pl_ds = xr.Dataset(data_vars=pl_data, coords=pl_coords)
print(f"  Variables: {list(pl_ds.data_vars)}, Dims: {dict(pl_ds.dims)}")

# Fix coordinates
sfc_ds = fix_coordinates(sfc_ds)
pl_ds  = fix_coordinates(pl_ds)

print("\n✓ All datasets loaded successfully")

In [0]:
# ------------------------------
# Prepare variables 
# ------------------------------
# Surface variables
sfc_ds_renamed = sfc_ds.rename({"t2m": "2t", "u10": "10u", "v10": "10v"})
surf_vars = {
    "2t": torch.from_numpy(sfc_ds_renamed["2t"].to_numpy()[None, None]),
    "10u": torch.from_numpy(sfc_ds_renamed["10u"].to_numpy()[None, None]),
    "10v": torch.from_numpy(sfc_ds_renamed["10v"].to_numpy()[None, None]),
    "msl": torch.from_numpy(sfc_ds_renamed["msl"].to_numpy()[None, None]),
}

# Pressure-level variables
levels = np.array([1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50])
atmos_vars = {
    "t": torch.from_numpy(pl_ds["t"].sel(isobaricInhPa=list(levels)).to_numpy()[None, None]),
    "u": torch.from_numpy(pl_ds["u"].sel(isobaricInhPa=list(levels)).to_numpy()[None, None]),
    "v": torch.from_numpy(pl_ds["v"].sel(isobaricInhPa=list(levels)).to_numpy()[None, None]),
    "q": torch.from_numpy(pl_ds["q"].sel(isobaricInhPa=list(levels)).to_numpy()[None, None]),
    "z": torch.from_numpy(pl_ds["z"].sel(isobaricInhPa=list(levels)).to_numpy()[None, None]),
}

# Metadata (timestamp auto-extracted from the data in Cell 10)
metadata = Metadata(
    lat=torch.from_numpy(pl_ds.latitude.values),
    lon=torch.from_numpy(pl_ds.longitude.values),
    time=(data_time,),  # auto-extracted from the NetCDF time variable
    atmos_levels=levels,
)

batch = Batch(
    surf_vars=surf_vars,
    static_vars={k: torch.from_numpy(v) for k, v in static_vars.items()},
    atmos_vars=atmos_vars,
    metadata=metadata,
)

In [0]:
# Check available memory before inference
memory_info = psutil.virtual_memory()
available_gb = memory_info.available / (1024**3)
print(f"Available memory: {available_gb:.2f} GB")


In [0]:
# ------------------------------
# Load model + run inference
# ------------------------------
print("Running Aurora inference (single step)...")
model.eval()
model = model.to("cuda")

with torch.inference_mode():
    preds = [p.to("cpu") for p in rollout(model, batch, steps=1)]
    torch.cuda.empty_cache()

print("✓ Inference complete")

In [0]:
# ------------------------------
# Save output 
# ------------------------------
print("Saving forecast...")
batch_step = preds[0]

lat = batch_step.metadata.lat.cpu().numpy()
lon = lon_360_to_180(batch_step.metadata.lon.cpu().numpy())

# Atmospheric variables
atm_data = {v: (("isobaricInhPa", "latitude", "longitude"),
                 np.squeeze(t.cpu().numpy()).astype(np.float32))
            for v, t in batch_step.atmos_vars.items()}

# Surface variables
surf_data = {v: (("latitude", "longitude"),
                  np.squeeze(t.cpu().numpy()).astype(np.float32))
             for v, t in batch_step.surf_vars.items()}

# Static variables
static_data = {}
for v, t in batch_step.static_vars.items():
    var_name = "z_sfc" if v == "z" else v
    static_data[var_name] = (("latitude", "longitude"),
                              t.cpu().numpy().astype(np.float32))

# Combine all
combined_vars = {**atm_data, **surf_data, **static_data}

ds = xr.Dataset(
    data_vars=combined_vars,
    coords={
        "time": ("time", [metadata.time[0]]),
        "valid_time": ("time", [metadata.time[0]]),
        "isobaricInhPa": ("isobaricInhPa", levels),
        "latitude": ("latitude", lat),
        "longitude": ("longitude", lon),
    },
)

out_path = os.path.join(output_dir, "aurora_forecast_single.nc")
ds.to_netcdf(out_path)

print(f"✓ Saved forecast: aurora_forecast_single.nc")
print("\nDone!")